# Human Review — NZ Moment Tensor Catalogue

> **Made by Claude. Not tested.** This notebook is indicative — it was
> generated to hand over a working recipe, but no cell has been executed
> end-to-end. Expect to troubleshoot: read error messages, check paths,
> and ask questions. Fixing it is part of the learning.

**Audience**: final-year undergraduate and masters students.

**The review follows four steps, mirroring the real pipeline:**
1. **Map** — the event, the station network, and a search radius YOU
   control;
2. **Waveform processing** — raw counts → displacement → filtered,
   rotated, inversion-ready records;
3. **TDMT inversion** — full manual control over **stations, time
   shifts (zcor), depth, record length, and velocity model**;
4. **Surface displacements** — the Okada forward model of YOUR
   solution: would this earthquake be visible to InSAR?

You then decide whether you can do better than the automated pipeline —
and publish
your reviewed solution into the shared human catalogue
(`events_human/catalogue_human.csv`), which many students build
together, one reviewed event at a time.

**Ground rules**
1. The automated archive `events/` is **read-only**. Your work goes in
   `events_human/` only.
2. Never run publishing scripts (`run03_publish.py`, anything emailing).
3. Every solution you publish carries your name and your reasoning.

**Read first** (in this repository):
- `docs/METHOD.md` — how the automated pipeline works.
- `docs/REVIEW_LEARNINGS.md` — the reviewer watch-list: grid-edge
  depths, layer-boundary VR spikes, noise fitting, coda contamination,
  "passenger" stations. **Read it twice.**
- Ristau (2008), SRL 79(3) — the NZ velocity models used here.
- https://eq.comoglu.com/bb — interactive beachball intuition-builder.

**Start with `2026p091845`** (M4.1, near Seddon) unless your
supervisor assigns another. Why it's a great playground:
- clean impulsive signal at many stations across a wide azimuth range;
- the automated run used only 3 stations (BSWZ, WMVZ, WLCZ) but benched
  several "yellow" stations that *individually fit well* — THZ, MRZ,
  WRRZ, KHZ, WEL — so there is real room to test whether adding them
  improves or degrades the solution;
- its depth (automated 12 km vs GeoNet 8.8 km) is worth interrogating.
Other good candidates from the 2026 review: `2026p140531` (7 stations,
DC 99, offshore Fiordland) and `2026p221690` (small but clean).


## 0. Installation (one-time, ~30 min)

Do this in a terminal; come back to the notebook afterwards.

```bash
# 1. Fork the repository on GitHub (button top-right), then:
git clone https://github.com/<YOUR-USERNAME>/auto_tdmt_NZ.git
cd auto_tdmt_NZ

# 2. Install pixi (the package manager) if you don't have it:
curl -fsSL https://pixi.sh/install.sh | bash
# restart your terminal after this

# 3. Install the project environment (Python, ObsPy, mttime, ...):
pixi install

# 4. Add Jupyter to the environment:
pixi add notebook ipywidgets

# 5. Green's functions (precomputed — you never run CPS):
#    download the tarball attached to the "gf-latest" release on the
#    ORIGINAL repository (github.com/Dani-Lindsay/auto_tdmt_NZ ->
#    Releases -> gf-latest) and extract it:
mkdir -p ~/mt_review/gf_library
tar -xzf ~/Downloads/<the-gf-tarball>.tar.gz -C ~/mt_review/gf_library
# GF_DIR below must be the directory that DIRECTLY contains
# nz_south_ristau2008/ and nz_north_ristau2008/ — check with `ls`.

# 6. Launch the notebook:
pixi run jupyter notebook human_review.ipynb
```

**Prefer VS Code?** You can run this notebook there instead of the
browser: install the **Jupyter** and **Python** extensions, add the
kernel package once (`pixi add ipykernel` in the repo), open
`human_review.ipynb`, then click **Select Kernel** (top right) →
*Python Environments* → choose the interpreter at
`.pixi/envs/default/bin/python` inside this repository. Run cells with
Shift+Enter. Everything else in this notebook works identically.

Waveforms download automatically from GeoNet (CC BY 3.0 NZ — credit
GeoNet in anything you produce) and are cached locally after first use.


In [ ]:
# ============================================================
# 1. STUDENT CONFIGURATION — edit these four lines, run this cell FIRST
# ============================================================
REVIEWER = "Your Name <you@university.ac.nz>"   # <-- EDIT: you
EVENT_ID = "2026p091845"                        # <-- EDIT: event to review
GF_DIR   = "~/mt_review/gf_library"             # <-- EDIT: extracted GFs
WORKDIR  = "~/mt_review/work"                   # scratch space (not in git)
RADIUS_KM = None       # <-- EDIT in Step 1: station search radius (km);
                       #     None = the magnitude-scaled default

# ---- nothing below needs editing ---------------------------------
import os, sys, json, shutil, warnings          # standard library tools
from pathlib import Path                        # object-oriented file paths
from datetime import date                       # for time-stamping your review

GF_DIR  = str(Path(GF_DIR).expanduser())        # expand "~" to your home dir
WORKDIR = Path(WORKDIR).expanduser() / EVENT_ID # one scratch dir per event
WORKDIR.mkdir(parents=True, exist_ok=True)      # create it if missing

# the pipeline reads these environment variables when it is imported,
# so they MUST be set before the imports below
os.environ["AUTO_TDMT_GF"] = GF_DIR             # where Green's functions live
os.environ["AUTO_TDMT_EVENTS"] = str(WORKDIR / "auto_rerun")  # rerun output

REPO = Path.cwd()                               # notebook runs from repo root
assert (REPO / "config.py").exists(), \
    "Run the notebook from the auto_tdmt_NZ repository root"
sys.path.insert(0, str(REPO))                   # make pipeline importable
warnings.filterwarnings("ignore")               # silence obspy chatter

import config      # all thresholds and paths
import waveforms   # download + pre-processing
import greens      # Green's function staging
import invert      # mtinv.in writing, mttime driving, summaries
from geonet import get_event                    # GeoNet event metadata

# sanity checks with helpful messages (fail loud, never guess)
assert Path(GF_DIR).exists(), f"GF library not found at {GF_DIR}"
for m in config.GF_MODELS:                      # both NZ velocity models
    assert (Path(GF_DIR) / m).exists(), \
        f"model {m} missing under {GF_DIR} — check tarball extraction"
print("environment OK — GF models available:", config.GF_MODELS)


## Orientation A — browse the automated catalogue

Pick events where a human adds value: grade C/D with red flags from the
watch-list, or B's you suspect could be A's with a better station set.


In [ ]:
import pandas as pd                              # tables

cat = pd.read_csv(REPO / "events" / "catalogue.csv")   # automated catalogue
print(f"{len(cat)} automated solutions; grades:",
      cat.Grade.value_counts().to_dict())              # grade distribution
# show the columns you care about when choosing an event to review
cat[["PublicID", "Date", "Mw", "Depth", "VR", "DC", "Grade",
     "NS", "AzGap", "quality_flag"]]


## Orientation B — the automated answer for your event, and its reasoning

Read what the machine did *and why* before touching anything. The
`*_station_waveforms_*.jpg` figure shows every candidate station with
its fate (black = used, orange = eliminated by fit, grey = rejected)
and the drop reason printed under each name — those orange "yellow
stations" are your playground. The `*_depth_sensitivity.jpg` shows
whether the depth sits on a believable plateau or a suspect spike.


In [ ]:
from IPython.display import Image, display      # inline images

auto_dir = config.find_event_dir(EVENT_ID, REPO / "events")  # locate archive
assert auto_dir, f"{EVENT_ID} not found in events/"
auto = json.loads((auto_dir / "solution.json").read_text())  # the solution
p, q = auto["preferred"], auto["quality"]        # preferred sol + quality

print(f"AUTOMATED: Mw {p['mw']:.2f}  depth {p['depth_km']:g} km  "
      f"VR {p['vr']:.1f}%  DC {p['pdc']:.0f}%  grade {q['grade']}  "
      f"gap {q['azimuthal_gap_deg']:.0f} deg  band {auto.get('chosen_band')}")
print("plane1 %(strike).0f/%(dip).0f/%(rake).0f" % p["plane1"])
print("used:", [r["station"] for r in auto["stations_used"]])
print("\ndropped (station | reason):")
for d in auto["stations_dropped"]:               # every exclusion, with reason
    print("  ", d["station"], "|", d["reason"])
for jpg in sorted(auto_dir.glob("*.jpg")):       # display all figures
    display(Image(str(jpg), width=900))


## Step 1 — the map: event, stations, and YOUR search radius

Everything starts with geometry: where is the event, where are the
stations, and how far out is it worth looking? The automated pipeline
scales the radius with magnitude (120–300 km); here YOU choose. Edit
`RADIUS_KM`, rerun the cell, and watch which stations enter the pool —
then think about azimuth coverage (stations all on one side cannot
constrain the mechanism well) before moving on. Your radius carries
through the rest of the notebook.


In [ ]:
import numpy as np                               # arrays
import matplotlib.pyplot as plt                   # plotting
import cartopy.crs as ccrs                        # map projections
import cartopy.feature as cfeature                # coastlines, ocean, land
from obspy import UTCDateTime                     # time handling
from geonet import fdsn_client                    # GeoNet FDSN client

# apply YOUR radius to the whole pipeline (the pipeline asks this
# function for the radius, so overriding it here affects every later
# step — a deliberate, visible monkeypatch)
if RADIUS_KM is not None:                         # None keeps the default
    config.station_max_dist_km = lambda mag: float(RADIUS_KM)
USE_RADIUS = config.station_max_dist_km(EV.prelim_mag)  # what's in force

origin = UTCDateTime(EV.origin_time)              # event origin time
client = fdsn_client(origin)                      # NRT or archive service
# ask for stations out to well beyond the radius, so you can SEE what
# a bigger radius would add (map shows both inside and outside)
_, all_rows = waveforms.select_stations(client, EV, origin,
                                        max_dist_km=USE_RADIUS + 150)

fig = plt.figure(figsize=(8, 9))                  # the map figure
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mercator())
pad = (USE_RADIUS + 180) / 111.0                  # degrees of padding
ax.set_extent([EV.longitude - pad, EV.longitude + pad,
               EV.latitude - pad, EV.latitude + pad])
ax.add_feature(cfeature.OCEAN, facecolor="#eaf4f8")   # style: ocean
ax.add_feature(cfeature.LAND, facecolor="#f5f2ec")    # land
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)     # coastline
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

inside = [r for r in all_rows if r["distance_km"] <= USE_RADIUS]
outside = [r for r in all_rows if r["distance_km"] > USE_RADIUS]
ax.plot([r["longitude"] for r in outside],        # beyond your radius
        [r["latitude"] for r in outside], "^", color="0.7",
        markersize=6, transform=ccrs.PlateCarree(), linestyle="none")
ax.plot([r["longitude"] for r in inside],         # inside = the pool
        [r["latitude"] for r in inside], "^", color="#E69F00",
        markeredgecolor="black", markersize=8,
        transform=ccrs.PlateCarree(), linestyle="none")
for r in all_rows:                                # name every station
    ax.annotate(r["station"], (r["longitude"], r["latitude"]),
                xytext=(3, 3), textcoords="offset points", fontsize=6,
                xycoords=ccrs.PlateCarree()._as_mpl_transform(ax))
ax.plot(EV.longitude, EV.latitude, "*", color="#D55E00",
        markeredgecolor="black", markersize=18,
        transform=ccrs.PlateCarree())             # the epicentre
theta = np.linspace(0, 2 * np.pi, 200)            # the radius circle,
circ_lat = EV.latitude + USE_RADIUS / 111.0 * np.sin(theta)   # drawn as
circ_lon = EV.longitude + USE_RADIUS / (111.0 * np.cos(       # a small-
    np.radians(EV.latitude))) * np.cos(theta)                 # circle
ax.plot(circ_lon, circ_lat, "-", color="#D55E00", linewidth=1.2,
        transform=ccrs.PlateCarree())
ax.set_title(f"{EVENT_ID}  M{EV.prelim_mag:.1f}  "
             f"search radius {USE_RADIUS:g} km  "
             f"({len(inside)} stations in pool)")
plt.show()

# azimuth coverage check: which 45-degree sectors are occupied?
sectors = sorted({int(r["azimuth"] // 45) for r in inside})
print(f"occupied azimuth sectors (of 8): {sectors} "
      f"-> think about the gaps before continuing")


## Step 2 — waveform processing, from raw counts to inversion-ready

Before inverting anything, see what happens to a seismogram on its way
into the inversion. `fetch_and_process` can record three snapshots per
station when given a `stages` dictionary:

- **raw** — what the sensor recorded (digital counts);
- **displacement** — after detrending and removing the instrument
  response (now ground displacement in metres, still N/E/Z);
- **final** — after rotation to radial/transverse, bandpass filtering,
  decimation to 1 sample/s, trimming to origin−30 s → +200 s, and
  conversion to cm (the TDMT convention).

The cell below fetches ALL candidate stations for your chosen band and
plots the three stages for one station. Change `SHOW_STATION` and rerun
to inspect others. This also gives you `pool` — every available station
with its peak-to-noise quality (`pk_n`) and tier — which you'll use to
choose your station set in the next section.


In [ ]:
BAND = (0.02, 0.10)      # filter band in Hz: 0.02-0.10 Hz = 10-50 s period
# (10-50 s is right for M<4.5; try (0.02, 0.05) = 20-50 s for larger)

stages = {}                                       # dict -> snapshots recorded
prep_dir = WORKDIR / f"prep_{config.band_tag(BAND)}"   # scratch for SAC files
EV = get_event(EVENT_ID)                          # event metadata from GeoNet
pool, prep_dropped = waveforms.fetch_and_process( # download + pre-process ALL
    EV, prep_dir, BAND, stages=stages)            # candidates; record stages

# table of every usable station: name, distance, azimuth, quality tier
print(f"{'station':8s} {'km':>5s} {'az':>4s} {'pk/n':>6s}  tier")
for r in sorted(pool, key=lambda r: r["distance_km"]):
    print(f"{r['station']:8s} {r['distance_km']:5.0f} {r['azimuth']:4.0f} "
          f"{r.get('pk_n', float('nan')):6.1f}  {r.get('tier','?')}")
print("\nnot usable (station | reason):")
for d in prep_dropped:
    print("  ", d["station"], "|", d["reason"][:70])


In [ ]:
SHOW_STATION = "BSWZ"     # <-- EDIT: which station's processing to display

import matplotlib.pyplot as plt                   # plotting

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=False)
for ax, key, label in zip(
        axes,
        ["raw", "displacement", "final"],
        ["raw (counts)", "response removed (displacement, m)",
         "rotated ZRT + filtered + 1 sps + trimmed (cm)"]):
    st = stages[key].select(station=SHOW_STATION) # this station's snapshot
    for tr in st:                                 # one line per component
        t = tr.times()                            # seconds from trace start
        ax.plot(t, tr.data, lw=0.7,
                label=tr.stats.channel)           # channel = e.g. HHZ or R/T
    ax.set_ylabel(label, fontsize=8)
    ax.legend(loc="upper right", fontsize=7)
axes[-1].set_xlabel("time (s)")
fig.suptitle(f"{SHOW_STATION}: raw -> displacement -> inversion-ready")
plt.tight_layout(); plt.show()


## Step 3 — the TDMT inversion: five knobs, all manual

`invert_event()` below writes the mttime control file (`mtinv.in`)
yourself-style and runs the inversion. Everything the automation
decides is now YOUR decision:

| knob | meaning | example |
|---|---|---|
| `stations` | exactly which stations to invert — edit this list | `["BSWZ","WMVZ","WLCZ","THZ","MRZ"]` |
| `depths` | depth grid to search (km); a single value fixes the depth | `[6,8,10,12,14]` or `[8]` |
| `band` | filter band (Hz), data AND Green's functions | `(0.02, 0.10)` |
| `model` | 1-D velocity model for the Green's functions | `"nz_south_ristau2008"` |
| `record_length_s` | seconds of each record to fit (from origin−30 s) | `100` (or `None` = distance-adaptive) |
| `zcor` | time shifts: `"auto"` lets mttime cross-correlate; or give seconds per station to shift manually | `{"THZ": 2, "MRZ": -1}` |

**Depth tip**: if you have a relocated catalogue (e.g. a
double-difference or template-matching study) with a well-constrained
depth for your event, FIX the depth to it (`depths=[that_value]`) and
tweak the other knobs to get the best solution at the *right* depth —
that is often more defensible than letting a 3-station VR curve choose.

**Yellow-station exercise** (the point of this event): start from the
automated station set, then add the benched good-lookers one at a time
(for 2026p091845 try THZ, MRZ, WRRZ, KHZ, WEL) and watch VR, DC and the
mechanism. A station that barely moves the mechanism but widens azimuth
coverage is usually worth keeping; one that rotates it wildly or
craters VR is telling you something — work out which watch-list item
applies.


In [ ]:
import numpy as np                                # arrays
import pandas as _pd                              # mtinv.in station table

def invert_event(stations,                        # list of station codes
                 depths,                          # list of depths (km)
                 band=BAND,                       # (fmin, fmax) in Hz
                 model="nz_south_ristau2008",     # velocity model name
                 record_length_s=None,            # None = distance-adaptive
                 zcor="auto",                     # "auto" or {station: s}
                 tag="run"):                      # label for the scratch dir
    """Run one mttime inversion with fully manual choices."""
    wd = WORKDIR / f"{tag}_{config.band_tag(band)}"   # scratch dir per run
    wd.mkdir(parents=True, exist_ok=True)
    # -- 1. data: reuse the pre-processed pool if band matches, else refetch
    if band == BAND:
        rows_all = pool                            # from section 4
        src_dir = prep_dir
    else:
        rows_all, _ = waveforms.fetch_and_process(EV, wd, band)
        src_dir = wd
    # -- 2. select exactly the stations YOU asked for
    rows = [r for r in rows_all if r["station"] in set(stations)]
    missing = set(stations) - {r["station"] for r in rows}
    assert not missing, f"not in the usable pool: {sorted(missing)}"
    rows.sort(key=lambda r: r["distance_km"])      # mttime convention
    # copy the SAC files into this run's dir (mtinv paths are relative)
    for r in rows:
        sid = f"{r['network']}.{r['station']}.{r['location']}"
        for comp in "ZRT":
            shutil.copy(src_dir / f"{sid}.{comp}.dat", wd)
    # -- 3. stage bandpassed Green's functions for these stations/depths
    greens.stage_event_greens(model, rows, list(depths), band, wd / "greens")
    # -- 4. build the mtinv.in control file (this IS the inversion setup —
    #       read the printed file and understand every column)
    tail = config.window_tail_s(EV.prelim_mag)     # magnitude-dependent tail
    npts = []                                      # fitted samples per station
    for r in rows:
        if record_length_s is not None:            # your manual choice...
            n = int(record_length_s)
        else:                                      # ...or the adaptive rule
            n = int(min(config.INV_NPTS, max(
                config.WINDOW_MIN_S,
                config.TIME_BEFORE_S
                + r["distance_km"] / config.WINDOW_GROUP_VEL_KMS + tail)))
        npts.append(min(n, 230))                   # cannot exceed the data
    ts = []                                        # alignment start (samples)
    for r in rows:                                 # base = 30 s pre-origin
        shift = 0 if zcor == "auto" else int(round(zcor.get(r["station"], 0)))
        ts.append(config.TIME_BEFORE_S + shift)    # your shift moves the data
    frame = _pd.DataFrame({
        "station": [f"{r['network']}.{r['station']}.{r['location']}"
                    for r in rows],                # SAC file basename
        "distance": [r["gf_distance_km"] for r in rows],  # GF grid distance
        "azimuth": [round(r["azimuth"], 2) for r in rows],
        "ts": ts,                                  # where fitting starts
        "npts": npts,                              # how many seconds to fit
        "dt": config.DT,                           # 1 sample per second
        "used": 1,                                 # 1 = include in inversion
        "longitude": [r["longitude"] for r in rows],
        "latitude": [r["latitude"] for r in rows],
    })
    headers = dict(datetime=EV.origin_time, longitude=EV.longitude,
                   latitude=EV.latitude,
                   depth=",".join(f"{d:.4f}" for d in depths),
                   path_to_data=".", path_to_green="greens",
                   green="herrmann", components="ZRT",
                   degree=5,                       # 5 = deviatoric MT
                   weight="distance",              # inverse-distance weights
                   plot=0,
                   correlate=1 if zcor == "auto" else 0)  # solve shifts?
    with open(wd / "mtinv.in", "w") as f:          # write the control file
        for k, v in headers.items():
            f.write(f"{k:<15}{v}\n")
        f.write(frame.to_string(index=False))
    print(open(wd / "mtinv.in").read())            # READ THIS — it is the
    # -- 5. run mttime                             #   whole inversion setup
    cwd = os.getcwd(); os.chdir(wd)                # mtinv paths are relative
    try:
        inv = invert.run_inversion(wd / "mtinv.in")
        inv.plot(view="waveform", option="preferred",
                 format="jpg", show=False)         # writes bbwaves*.jpg
    finally:
        os.chdir(cwd)                              # always restore
    # -- 6. summarise the preferred solution
    mt = inv.moment_tensors[inv.preferred_tensor_id]
    sdr = tuple(float(x) for x in mt.fps[0][:3])   # strike/dip/rake plane 1
    print(f"\n[{tag}] depth {float(mt.depth):g} km  Mw {mt.mw:.2f}  "
          f"VR {float(mt.total_VR):.1f}%  DC {float(mt.pdc):.0f}%  "
          f"plane1 {sdr[0]:.0f}/{sdr[1]:.0f}/{sdr[2]:.0f}")
    for r in mt.station_table.itertuples():        # per-station fit quality
        print(f"    {r.station}: station VR {float(r.VR):.0f}")
    display(Image(str(sorted(wd.glob('bbwaves*.jpg'))[0]), width=900))
    return inv, rows, wd                           # keep for comparison/publish

def preferred(inv):                                # tiny convenience helper
    return inv.moment_tensors[inv.preferred_tensor_id]

def compare(inv_a, inv_b):                         # mechanism rotation between
    a = tuple(float(x) for x in preferred(inv_a).fps[0][:3])  # two runs,
    b = tuple(float(x) for x in preferred(inv_b).fps[0][:3])  # flip-proof
    ang = invert.tensor_angle_deg(a, b)
    print(f"mechanism rotation between runs: {ang:.1f} deg")
    return ang

def depth_profile(inv):                            # quick VR/DC vs depth plot
    zz = [float(mt.depth) for mt in inv.moment_tensors]
    vr = [float(mt.total_VR) for mt in inv.moment_tensors]
    dc = [float(mt.pdc) for mt in inv.moment_tensors]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(zz, vr, "ko-"); ax[0].set(xlabel="depth km", ylabel="VR %")
    ax[1].plot(zz, dc, "ko-"); ax[1].set(xlabel="depth km", ylabel="DC %")
    plt.tight_layout(); plt.show()

print("sandbox ready — invert_event(stations=[...], depths=[...])")


In [ ]:
# --- run 1: reproduce the automated station set --------------------
stations = ["BSWZ", "WMVZ", "WLCZ"]        # <-- the machine's choice
inv0, rows0, wd0 = invert_event(
    stations=stations,
    depths=[4, 6, 8, 10, 12, 14, 16],      # search around GeoNet's 8.8 km
    tag="auto_set")
depth_profile(inv0)                        # is the depth a plateau or spike?


In [ ]:
# --- run 2: YOUR station set — add the yellow stations one by one --
stations = ["BSWZ", "WMVZ", "WLCZ",        # keep the core...
            "THZ", "MRZ"]                  # <-- EDIT: benched good-lookers
inv1, rows1, wd1 = invert_event(
    stations=stations,
    depths=[4, 6, 8, 10, 12, 14, 16],
    tag="with_yellows")
compare(inv0, inv1)                        # did the mechanism move?
depth_profile(inv1)


In [ ]:
# --- run 3 (optional): fix the depth from a relocated catalogue ----
# If a relocation study gives this event a well-constrained depth, fix
# it and tune everything else at that depth:
inv2, rows2, wd2 = invert_event(
    stations=["BSWZ", "WMVZ", "WLCZ", "THZ", "MRZ"],
    depths=[9],                            # <-- the relocated depth (km)
    record_length_s=100,                   # <-- your record-length choice
    zcor="auto",                           # or e.g. {"THZ": 2} seconds
    tag="fixed_depth")
compare(inv1, inv2)


## Step 4 — surface displacements: would InSAR see it?

The reason this pipeline exists: convert YOUR moment tensor into a
predicted surface displacement field (Okada 1985 dislocation model,
Wells & Coppersmith 1994 fault dimensions) and ask whether the
deformation could be measured from space. Both nodal planes are
modelled — the far-field pattern is identical, so only near-field
detail differs between them. The 1 cm contour is roughly the InSAR
detectability threshold used by the pipeline.


In [ ]:
# build a full solution dict from YOUR final inversion run
SOLUTION = invert.summarize(FINAL_INV, EV, FINAL_ROWS,
                            [],                    # no drop record here
                            FINAL_MODEL)           # your velocity model
SOLUTION["filter_band_hz"] = list(FINAL_BAND)      # provenance
SOLUTION["quality"] = invert.quality_gates(SOLUTION)   # grade it

import okada_forward                               # the forward model
fwd = okada_forward.forward_both_planes(SOLUTION)  # both nodal planes
print(f"peak predicted displacement: {fwd['peak_abs_m']*100:.2f} cm  "
      f"-> detectable by InSAR: {fwd['detectable']}")

try:                                               # colour-blind-safe
    from cmcrameri import cm as _cmc               # diverging colormap
    CMAP = _cmc.vik
except ImportError:                                # graceful fallback
    CMAP = "RdBu_r"

fig, axes = plt.subplots(2, 3, figsize=(13, 8),
                         sharex=True, sharey=True)
vmax = max(abs(fwd[pl][c]).max()                   # one symmetric scale
           for pl in ("plane1", "plane2")
           for c in ("ue_m", "un_m", "uz_m")) * 100
for i, pl in enumerate(("plane1", "plane2")):      # one row per plane
    for j, (comp, label) in enumerate(
            (("ue_m", "east"), ("un_m", "north"), ("uz_m", "up"))):
        ax = axes[i, j]
        u_cm = fwd[pl][comp] * 100                 # metres -> cm
        pm = ax.pcolormesh(fwd[pl]["x_km"], fwd[pl]["y_km"], u_cm,
                           cmap=CMAP, vmin=-vmax, vmax=vmax)
        outline = okada_forward.fault_outline(fwd[pl]["fault"])
        ax.plot(outline["outline_x_km"],           # fault rectangle
                outline["outline_y_km"], "k-", lw=1)
        ax.plot(outline["top_x_km"], outline["top_y_km"],
                "-", color="#E69F00", lw=2.5)      # up-dip (shallow) edge
        ax.set_title(f"{pl} {label}", fontsize=9)
        ax.set_aspect("equal")
axes[1, 1].set_xlabel("east (km)"); axes[0, 0].set_ylabel("north (km)")
fig.colorbar(pm, ax=axes, shrink=0.7, label="displacement (cm)")
fig.suptitle(f"{EVENT_ID}: Okada surface displacement from YOUR "
             f"solution (Mw {SOLUTION['preferred']['mw']:.2f})")
plt.show()


## Step 5 — decide and document

Be honest and specific. `decision` is one of:
- `"accept_automated"` — the machine's answer stands; your review is the
  human confirmation (valuable!).
- `"revised"` — your solution replaces it in the human catalogue; state
  exactly what you changed and *why* (name the watch-list item).
- `"reject"` — no defensible solution exists (also valuable — say why).


In [ ]:
REVIEW = {
    "reviewer": REVIEWER,                          # who
    "date": str(date.today()),                     # when
    "decision": "revised",       # accept_automated | revised | reject
    "changes": "added THZ and MRZ (benched with station VR 54-65); "
               "depth fixed to relocated 9 km",    # WHAT you changed
    "notes": "azimuth gap 159->98; mechanism rotated only 6 deg; "
             "VR down 3 points but coverage worth it",  # WHY it's better
}
# which run is your final answer? point these at it:
FINAL_INV, FINAL_ROWS, FINAL_WD = inv1, rows1, wd1
FINAL_MODEL, FINAL_BAND = "nz_south_ristau2008", BAND
print(json.dumps(REVIEW, indent=2))


## Step 6 — stage your solution in `events_human/`

This writes `events_human/<same-dir-name>/` (your `solution.json` with
a `human_review` block, your waveform-fit figure, plus the automated
figures as `auto_*.jpg` for comparison) and appends **one row to the
shared master catalogue** `events_human/catalogue_human.csv`, whose
columns mirror the automated `events/catalogue.csv` so the two can be
compared side by side. Many students append to the same file — if your
Pull Request later shows a conflict in this CSV, `git pull --rebase`
and re-run this cell. Reruns replace only YOUR row (same event + same
reviewer); other reviewers' rows for the same event are kept — multiple
reviews of one event by different authors are welcome, and disagreement
between reviewers is itself useful information.


In [ ]:
# reuse the solution dict built in Step 4 and attach your review
human = SOLUTION                                   # from Step 4 (summarised)
human["human_review"] = REVIEW                     # who/what/why
human["automated_reference"] = {                   # machine answer, for diff
    "solution_dir": auto_dir.name, "mw": p["mw"],
    "depth_km": p["depth_km"], "vr": p["vr"], "pdc": p["pdc"],
    "grade": q["grade"]}

hdir = REPO / "events_human" / auto_dir.name       # mirror the dir name
hdir.mkdir(parents=True, exist_ok=True)
(hdir / "solution.json").write_text(json.dumps(human, indent=2))
for src in sorted(FINAL_WD.glob("bbwaves*.jpg")):  # your fits figure
    shutil.copy(src, hdir / f"{EVENT_ID}_human_waveform_fits.jpg")
for src in sorted(auto_dir.glob("*.jpg")):         # machine figures, prefixed
    shutil.copy(src, hdir / f"auto_{src.name}")

# ---- append to the SHARED master catalogue (columns mirror events/) ----
hp, hq = human["preferred"], human["quality"]
row = {
    "PublicID": EVENT_ID,
    "Date": EV.origin_time[:10],
    "Latitude": EV.latitude, "Longitude": EV.longitude,
    "strike1": round(hp["plane1"]["strike"]), "dip1": round(hp["plane1"]["dip"]),
    "rake1": round(hp["plane1"]["rake"]),
    "strike2": round(hp["plane2"]["strike"]), "dip2": round(hp["plane2"]["dip"]),
    "rake2": round(hp["plane2"]["rake"]),
    "GeoNet_M": round(EV.prelim_mag, 2), "GeoNet_depth": round(EV.depth_km, 1),
    "Mw": round(hp["mw"], 2), "Depth": hp["depth_km"],
    "NS": hq["n_stations_used"], "AzGap": hq["azimuthal_gap_deg"],
    "Grade": hq["grade"], "DC": round(hp["pdc"]), "VR": round(hp["vr"], 1),
    "Band": config.band_tag(FINAL_BAND), "Model": FINAL_MODEL,
    # human-review columns (extra relative to the automated catalogue):
    "Reviewer": REVIEWER, "ReviewDate": REVIEW["date"],
    "Decision": REVIEW["decision"], "Changes": REVIEW["changes"],
    "Auto_Mw": p["mw"], "Auto_Depth": p["depth_km"], "Auto_Grade": q["grade"],
}
cat_path = REPO / "events_human" / "catalogue_human.csv"
hcat = pd.read_csv(cat_path) if cat_path.exists() else pd.DataFrame()
if len(hcat):        # replace only MY row for this event if I rerun this
    hcat = hcat[~((hcat["PublicID"] == EVENT_ID)   # cell; OTHER reviewers'
                  & (hcat["Reviewer"] == REVIEWER))]  # rows are kept —
    # multiple reviews of one event by different authors are welcome
hcat = pd.concat([hcat, pd.DataFrame([row])], ignore_index=True)
hcat = hcat.sort_values("PublicID")                # stable order = fewer
hcat.to_csv(cat_path, index=False)                 #   merge conflicts
print("staged:", hdir)
print(f"human catalogue now holds {len(hcat)} reviewed events")


## Step 7 — push your review to GitHub

Your changes must touch **only** `events_human/`. From a terminal in
the repository root:

```bash
git checkout -b review/<eventID>-<yourname>
git status                  # MUST list only events_human/ files
git add events_human/
git commit -m "human review <eventID>: <one-line summary> (<your name>)"
git push -u origin review/<eventID>-<yourname>
```

Open a **Pull Request** from your fork/branch to the main repository.
Paste your `REVIEW` block into the PR description, plus one sentence on
what the automated pipeline should learn from this event. The
maintainer merges — that is your solution entering the shared human
catalogue. PRs touching anything outside `events_human/` are closed
unmerged. If the catalogue CSV conflicts (another student merged
first): `git pull --rebase origin main`, re-run the staging cell above,
commit again.

## Troubleshooting (start here before asking)

- **Import errors** → not running from the repo root, or `pixi install`
  incomplete.
- **GF assertion** → `GF_DIR` must DIRECTLY contain the model folders.
- **`not in the usable pool`** → that station was dropped in
  pre-processing (see section 4's "not usable" list for the reason).
- **`No data available`** during download → normal; that station has no
  data for this event.
- **Everything fits terribly** → `docs/REVIEW_LEARNINGS.md` §1–2: is
  your event offshore/no-signal, coda-contaminated, or on a grid-edge
  depth? "No defensible solution" is a valid review outcome.

## Credits

Inversion: **mttime** (Chiang, LLNL). Green's functions: **CPS**
(Herrmann 2013) with **Ristau (2008)** NZ velocity models
(doi:10.1785/gssrl.79.3.400). Waveforms/events: **GeoNet** (CC BY 3.0
NZ). Method lineage: Dreger & Helmberger (1993), Dreger (2003), Minson
& Dreger (2008). Cite them, not this repository.

> **Made by Claude. Not tested.** — you were warned at the top; now you
> know enough to fix it.
